In [1]:
import pandas as pd
import glob
import os

# Path where your template files are stored
DATA_PATH = "../data/modeling_data"   # change if needed

# Find all template modeling files
file_paths = sorted(glob.glob(os.path.join(DATA_PATH, "template*_modeling.csv")))

print("Files detected:")
for f in file_paths:
    print(f)

# Load and combine
dfs = []
for path in file_paths:
    df = pd.read_csv(path)
    
    # Extract template name from filename
    template_name = os.path.basename(path).split("_")[0]  # template1, template2, ...
    df["source_template"] = template_name
    
    dfs.append(df)

# Concatenate into single modeling dataset
combined_df = pd.concat(dfs, ignore_index=True)

# Basic validation
print("\nShape of combined dataset:", combined_df.shape)
print("\nColumns:", combined_df.columns.tolist())
print("\nClass distribution:")
print(combined_df["IS_FRAUD"].value_counts())

# # Save combined dataset
# combined_df.to_csv("combined_modeling_data.csv", index=False)

# print("\nCombined dataset saved as: combined_modeling_data.csv")

# Save combined dataset
output_path = os.path.join(DATA_PATH, "combined_modeling_data.csv")
combined_df.to_csv(output_path, index=False)

print(f"\nCombined dataset saved to: {output_path}")

Files detected:
../data/modeling_data/template1_modeling.csv
../data/modeling_data/template2_modeling.csv
../data/modeling_data/template3_modeling.csv
../data/modeling_data/template4_modeling.csv
../data/modeling_data/template5_modeling.csv

Shape of combined dataset: (1000, 3)

Columns: ['TOTAL_NUM', 'IS_FRAUD', 'source_template']

Class distribution:
IS_FRAUD
False    841
True     159
Name: count, dtype: int64

Combined dataset saved to: ../data/modeling_data/combined_modeling_data.csv


In [12]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# ----------------------------
# 1) Load combined modeling data
# ----------------------------
DATA_FILE = "../data/modeling_data/combined_modeling_data.csv"   # change if needed
df = pd.read_csv(DATA_FILE)

# ----------------------------
# 2) Define features + target
# ----------------------------
# Uses TOTAL_NUM (numeric) + source_template (categorical)
X = df[["TOTAL_NUM", "source_template"]].copy()
# X = df["TOTAL_NUM"].copy()
y = df["IS_FRAUD"].astype(int)

# ----------------------------
# 3) 80/20 Train-Test Split (stratified)
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train fraud rate:", y_train.mean(), " Test fraud rate:", y_test.mean())

# ----------------------------
# 4) Preprocessing + Logistic Regression pipeline
# ----------------------------
numeric_features = ["TOTAL_NUM"]
categorical_features = ["source_template"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ],
    remainder="drop"
)

clf = Pipeline(steps=[
    ("prep", preprocess),
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",   # helps with imbalance
        solver="lbfgs"
    ))
])

# ----------------------------
# 5) Train
# ----------------------------
clf.fit(X_train, y_train)

# ----------------------------
# 6) Predict + Evaluate
# ----------------------------
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) if (fp + tn) else 0.0
fnr = fn / (fn + tp) if (fn + tp) else 0.0

print("\n--- Logistic Regression (80/20 split) ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")

print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

Train shape: (800, 2)  Test shape: (200, 2)
Train fraud rate: 0.17375  Test fraud rate: 0.175

--- Logistic Regression (80/20 split) ---
Accuracy : 0.8900
Precision: 0.6226
Recall   : 0.9429
F1       : 0.7500
ROC-AUC  : 0.9574
FPR      : 0.1212
FNR      : 0.0571

Confusion Matrix [ [TN FP], [FN TP] ]:
[[145  20]
 [  2  33]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9864    0.8788    0.9295       165
           1     0.6226    0.9429    0.7500        35

    accuracy                         0.8900       200
   macro avg     0.8045    0.9108    0.8397       200
weighted avg     0.9227    0.8900    0.8981       200



In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# ----------------------------
# 1) Load data
# ----------------------------
DATA_FILE = "../data/modeling_data/combined_modeling_data.csv"   # adjust path if needed
df = pd.read_csv(DATA_FILE)

# ----------------------------
# 2) Features + Target
# ----------------------------
X = df[["TOTAL_NUM", "source_template"]].copy()
y = df["IS_FRAUD"].astype(int)

# ----------------------------
# 3) 80–20 Stratified Split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ----------------------------
# 4) Preprocessing + Logistic Regression
# ----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), ["TOTAL_NUM"]),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), ["source_template"])
    ]
)

model = Pipeline([
    ("prep", preprocess),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

# Train model
model.fit(X_train, y_train)

# ----------------------------
# 5) Predict on test set
# ----------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Add predictions back to test dataframe
test_results = X_test.copy()
test_results["IS_FRAUD"] = y_test.values
test_results["y_pred"] = y_pred
test_results["y_proba"] = y_proba

# ----------------------------
# 6) Metrics per Template
# ----------------------------
for template in sorted(test_results["source_template"].unique()):

    subset = test_results[test_results["source_template"] == template]
    
    y_true = subset["IS_FRAUD"]
    y_pred_subset = subset["y_pred"]
    y_proba_subset = subset["y_proba"]

    acc = accuracy_score(y_true, y_pred_subset)
    prec = precision_score(y_true, y_pred_subset, zero_division=0)
    rec = recall_score(y_true, y_pred_subset, zero_division=0)
    f1 = f1_score(y_true, y_pred_subset, zero_division=0)

    # ROC-AUC only valid if both classes present
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_proba_subset)
    else:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred_subset, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0

    print(f"\n==============================")
    print(f"Template: {template}")
    print("==============================")

    print("\n--- Logistic Regression (80/20 split) ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print(f"FPR      : {fpr:.4f}")
    print(f"FNR      : {fnr:.4f}")

    print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_subset, digits=4))


Template: template1

--- Logistic Regression (80/20 split) ---
Accuracy : 0.7907
Precision: 0.5000
Recall   : 1.0000
F1       : 0.6667
ROC-AUC  : 1.0000
FPR      : 0.2647
FNR      : 0.0000

Confusion Matrix [ [TN FP], [FN TP] ]:
[[25  9]
 [ 0  9]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.7353    0.8475        34
           1     0.5000    1.0000    0.6667         9

    accuracy                         0.7907        43
   macro avg     0.7500    0.8676    0.7571        43
weighted avg     0.8953    0.7907    0.8096        43


Template: template2

--- Logistic Regression (80/20 split) ---
Accuracy : 0.9000
Precision: 0.7143
Recall   : 1.0000
F1       : 0.8333
ROC-AUC  : 1.0000
FPR      : 0.1333
FNR      : 0.0000

Confusion Matrix [ [TN FP], [FN TP] ]:
[[26  4]
 [ 0 10]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.8667    0.9286        30
           1     0

In [15]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# Create error column (1 = incorrect prediction, 0 = correct)
test_results["error"] = (test_results["IS_FRAUD"] != test_results["y_pred"]).astype(int)

# Create contingency table: Template × Error
contingency_table = pd.crosstab(
    test_results["source_template"],
    test_results["error"]
)

print("Contingency Table (Template × Error)")
print(contingency_table)

# Run Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("\nChi-Square Test Results")
print(f"Chi2 Statistic : {chi2:.4f}")
print(f"Degrees of Freedom : {dof}")
print(f"P-value : {p_value:.6f}")

# Interpretation
alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject H0 → Performance differs significantly across templates.")
else:
    print("\nResult: Fail to reject H0 → No significant performance difference across templates.")

Contingency Table (Template × Error)
error             0  1
source_template       
template1        34  9
template2        36  4
template3        38  1
template4        39  1
template5        31  7

Chi-Square Test Results
Chi2 Statistic : 12.2966
Degrees of Freedom : 4
P-value : 0.015277

Result: Reject H0 → Performance differs significantly across templates.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# ----------------------------
# 1) Load Data
# ----------------------------
DATA_FILE = "../data/modeling_data/combined_modeling_data.csv"   
df = pd.read_csv(DATA_FILE)

# ----------------------------
# 2) Define Features + Target
# ----------------------------
X = df[["TOTAL_NUM", "source_template"]].copy()
y = df["IS_FRAUD"].astype(int)

# ----------------------------
# 3) 80–20 Stratified Split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

# ----------------------------
# 4) Preprocessing + Random Forest Pipeline
# ----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), ["TOTAL_NUM"]),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), ["source_template"])
    ]
)

rf_model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

# ----------------------------
# 5) Train Model
# ----------------------------
rf_model.fit(X_train, y_train)

# ----------------------------
# 6) Predict + Evaluate
# ----------------------------
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_proba)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn) if (fp + tn) else 0.0
fnr = fn / (fn + tp) if (fn + tp) else 0.0

# ----------------------------
# 7) Print Results
# ----------------------------
print("\n--- Random Forest (80/20 split) ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")

print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

Train shape: (800, 2)  Test shape: (200, 2)

--- Random Forest (80/20 split) ---
Accuracy : 0.9650
Precision: 0.8947
Recall   : 0.9189
F1       : 0.9067
ROC-AUC  : 0.9460
FPR      : 0.0245
FNR      : 0.0811

Confusion Matrix [ [TN FP], [FN TP] ]:
[[159   4]
 [  3  34]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9815    0.9755    0.9785       163
           1     0.8947    0.9189    0.9067        37

    accuracy                         0.9650       200
   macro avg     0.9381    0.9472    0.9426       200
weighted avg     0.9654    0.9650    0.9652       200



In [23]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

# Predict once on entire test set
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

# Combine results into dataframe
test_results = X_test.copy()
test_results["IS_FRAUD"] = y_test.values
test_results["y_pred"] = y_pred
test_results["y_proba"] = y_proba

# Loop through each template
for template in sorted(test_results["source_template"].unique()):

    subset = test_results[test_results["source_template"] == template]

    y_true = subset["IS_FRAUD"]
    y_pred_subset = subset["y_pred"]
    y_proba_subset = subset["y_proba"]

    acc = accuracy_score(y_true, y_pred_subset)
    prec = precision_score(y_true, y_pred_subset, zero_division=0)
    rec = recall_score(y_true, y_pred_subset, zero_division=0)
    f1 = f1_score(y_true, y_pred_subset, zero_division=0)

    # ROC-AUC valid only if both classes exist
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_proba_subset)
    else:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred_subset, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0

    print("\n==============================")
    print(f"Template: {template}")
    print("==============================")

    print("\n--- Random Forest (80/20 split) ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print(f"FPR      : {fpr:.4f}")
    print(f"FNR      : {fnr:.4f}")

    print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_subset, digits=4))


Template: template1

--- Random Forest (80/20 split) ---
Accuracy : 0.9524
Precision: 0.8000
Recall   : 1.0000
F1       : 0.8889
ROC-AUC  : 1.0000
FPR      : 0.0588
FNR      : 0.0000

Confusion Matrix [ [TN FP], [FN TP] ]:
[[32  2]
 [ 0  8]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9412    0.9697        34
           1     0.8000    1.0000    0.8889         8

    accuracy                         0.9524        42
   macro avg     0.9000    0.9706    0.9293        42
weighted avg     0.9619    0.9524    0.9543        42


Template: template2

--- Random Forest (80/20 split) ---
Accuracy : 0.9756
Precision: 1.0000
Recall   : 0.9091
F1       : 0.9524
ROC-AUC  : 0.9439
FPR      : 0.0000
FNR      : 0.0909

Confusion Matrix [ [TN FP], [FN TP] ]:
[[30  0]
 [ 1 10]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9677    1.0000    0.9836        30
           1     1.0000    0.9

In [24]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

# Create error column
test_results["error_rf"] = (test_results["IS_FRAUD"] != test_results["y_pred"]).astype(int)

# Contingency table
cont_table_rf = pd.crosstab(
    test_results["source_template"],
    test_results["error_rf"]
)

print("Contingency Table (RF)")
print(cont_table_rf)

chi2, p_value, dof, expected = chi2_contingency(cont_table_rf)

print("\nChi-Square Test (Random Forest)")
print(f"Chi2 Statistic : {chi2:.4f}")
print(f"Degrees of Freedom : {dof}")
print(f"P-value : {p_value:.6f}")

Contingency Table (RF)
error_rf          0  1
source_template       
template1        40  2
template2        40  1
template3        38  0
template4        39  0
template5        36  4

Chi-Square Test (Random Forest)
Chi2 Statistic : 8.1311
Degrees of Freedom : 4
P-value : 0.086892


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
# If needed:
# pip install -U langgraph scikit-learn pandas numpy

import numpy as np
import pandas as pd

from typing import TypedDict, Optional, Dict, Tuple
from langgraph.graph import StateGraph, START, END  # LangGraph Graph API  [oai_citation:0‡docs.langchain.com](https://docs.langchain.com/oss/python/langgraph/graph-api?utm_source=chatgpt.com)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

# ----------------------------
# 1) Load data
# ----------------------------
DATA_FILE = "../data/modeling_data/combined_modeling_data.csv"
df = pd.read_csv(DATA_FILE)

X = df[["TOTAL_NUM", "source_template"]].copy()
y = df["IS_FRAUD"].astype(int).values

# ----------------------------
# 2) 80/20 stratified split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


# ----------------------------
# 3) Fit template-wise robust stats for the anomaly agent
# ----------------------------
def robust_stats(vals: np.ndarray) -> Tuple[float, float]:
    vals = np.asarray(vals, dtype=float)
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    return med, mad

template_stats: Dict[str, Tuple[float, float]] = {}
for t, g in X_train.assign(IS_FRAUD=y_train).groupby("source_template"):
    template_stats[t] = robust_stats(g["TOTAL_NUM"].values)

global_med, global_mad = robust_stats(X_train["TOTAL_NUM"].values)

# Global extreme thresholds (tails)
lo, hi = np.quantile(X_train["TOTAL_NUM"].astype(float), [0.005, 0.995])

# ----------------------------
# 4) Define LangGraph state + nodes (agents)
# ----------------------------
class FraudState(TypedDict, total=False):
    # Inputs
    total_num: float
    source_template: str

    # Agent outputs
    anomaly_score: float
    extreme_flag: int
    ml_proba: float

    # Orchestrator outputs
    final_proba: float
    final_pred: int


def anomaly_agent(state: FraudState) -> FraudState:
    """Template-aware robust anomaly score using MAD-based z."""
    t = state["source_template"]
    v = float(state["total_num"])

    med, mad = template_stats.get(t, (global_med, global_mad))
    if mad == 0 or np.isnan(mad):
        score = 0.0
    else:
        z = (v - med) / (1.4826 * mad)
        # Logistic mapping: near 0 when |z| < 3.5, increases beyond
        score = 1.0 / (1.0 + np.exp(-(abs(z) - 3.5)))

    return {"anomaly_score": float(score)}


def extreme_tail_agent(state: FraudState) -> FraudState:
    """Global tail detector (binary)."""
    v = float(state["total_num"])
    flag = int((v < lo) or (v > hi))
    return {"extreme_flag": flag}


def llm_fraud_agent(state: FraudState) -> FraudState:

    total = state["total_num"]
    template = state["source_template"]

    prompt = f"""
    You are an AI invoice fraud analyst.

    Evaluate whether the following invoice is fraudulent.

    Invoice information:
    - Invoice Amount: {total}
    - Invoice Template: {template}

    Typical fraud indicators:
    • Extremely high or low invoice amounts
    • Unusual template usage
    • Values inconsistent with normal transaction ranges

    Return ONLY a JSON object:

    {{
    "fraud_probability": number between 0 and 1,
    "reason": short explanation
    }}
    """

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    text = response.choices[0].message.content

    try:
        result = json.loads(text)
        prob = float(result["fraud_probability"])
    except:
        prob = 0.5

    return {"llm_proba": prob}


def orchestrator_agent(state: FraudState) -> FraudState:
    """
    Combine signals into final fraud probability.
    Keep ML as primary, but allow agent flags to adjust.
    """
    ml_p = float(state.get("ml_proba", 0.0))
    anom = float(state.get("anomaly_score", 0.0))
    extreme = int(state.get("extreme_flag", 0))

    # Weighted fusion (tuneable)
    final_p = 0.75 * ml_p + 0.20 * anom + 0.15 * extreme
    final_p = float(np.clip(final_p, 0.0, 1.0))

    pred = int(final_p >= 0.5)
    return {"final_proba": final_p, "final_pred": pred}


# ----------------------------
# 5) Build the LangGraph
# ----------------------------
graph = StateGraph(FraudState)
graph.add_node("anomaly_agent", anomaly_agent)
graph.add_node("extreme_tail_agent", extreme_tail_agent)
graph.add_node("llm_fraud_agent", llm_fraud_agent)
graph.add_node("orchestrator_agent", orchestrator_agent)

# Execution order: START -> (agents) -> orchestrator -> END
graph.add_edge(START, "anomaly_agent")
graph.add_edge("anomaly_agent", "extreme_tail_agent")
graph.add_edge("extreme_tail_agent", "llm_fraud_agent")
graph.add_edge("llm_fraud_agent", "orchestrator_agent")
graph.add_edge("orchestrator_agent", END)

app = graph.compile()

# ----------------------------
# 6) Run the agentic classifier on the test set
# ----------------------------
agentic_proba = []
agentic_pred = []

for _, row in X_test.iterrows():
    init_state: FraudState = {
        "total_num": float(row["TOTAL_NUM"]),
        "source_template": str(row["source_template"]),
    }
    out_state = app.invoke(init_state)
    agentic_proba.append(out_state["final_proba"])
    agentic_pred.append(out_state["final_pred"])

agentic_proba = np.array(agentic_proba, dtype=float)
agentic_pred = np.array(agentic_pred, dtype=int)

# ----------------------------
# 7) Evaluate
# ----------------------------
acc = accuracy_score(y_test, agentic_pred)
prec = precision_score(y_test, agentic_pred, zero_division=0)
rec = recall_score(y_test, agentic_pred, zero_division=0)
f1 = f1_score(y_test, agentic_pred, zero_division=0)
auc = roc_auc_score(y_test, agentic_proba)

cm = confusion_matrix(y_test, agentic_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) if (fp + tn) else 0.0
fnr = fn / (fn + tp) if (fn + tp) else 0.0

print("\n--- Agentic AI (LangGraph) (80/20 split) ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")

print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, agentic_pred, digits=4))


--- Agentic AI (LangGraph) (80/20 split) ---
Accuracy : 0.9950
Precision: 0.9697
Recall   : 1.0000
F1       : 0.9846
ROC-AUC  : 1.0000
FPR      : 0.0060
FNR      : 0.0000

Confusion Matrix [ [TN FP], [FN TP] ]:
[[167   1]
 [  0  32]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9940    0.9970       168
           1     0.9697    1.0000    0.9846        32

    accuracy                         0.9950       200
   macro avg     0.9848    0.9970    0.9908       200
weighted avg     0.9952    0.9950    0.9950       200



In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

# Combine test data + predictions
test_results = X_test.copy()
test_results["IS_FRAUD"] = y_test
test_results["agentic_pred"] = agentic_pred
test_results["agentic_proba"] = agentic_proba

# Loop through templates in TEST set only
for template in sorted(test_results["source_template"].unique()):

    subset = test_results[test_results["source_template"] == template]

    y_true = subset["IS_FRAUD"]
    y_pred_subset = subset["agentic_pred"]
    y_proba_subset = subset["agentic_proba"]

    acc = accuracy_score(y_true, y_pred_subset)
    prec = precision_score(y_true, y_pred_subset, zero_division=0)
    rec = recall_score(y_true, y_pred_subset, zero_division=0)
    f1 = f1_score(y_true, y_pred_subset, zero_division=0)

    # ROC-AUC only if both classes present
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_proba_subset)
    else:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred_subset, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0

    print("\n==============================")
    print(f"Template: {template}")
    print("==============================")

    print("\n--- Agentic AI (80–20 TEST SET) ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print(f"FPR      : {fpr:.4f}")
    print(f"FNR      : {fnr:.4f}")

    print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_subset, digits=4))


Template: template1

--- Agentic AI (80–20 TEST SET) ---
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
ROC-AUC  : 1.0000
FPR      : 0.0000
FNR      : 0.0000

Confusion Matrix [ [TN FP], [FN TP] ]:
[[37  0]
 [ 0  7]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        37
           1     1.0000    1.0000    1.0000         7

    accuracy                         1.0000        44
   macro avg     1.0000    1.0000    1.0000        44
weighted avg     1.0000    1.0000    1.0000        44


Template: template2

--- Agentic AI (80–20 TEST SET) ---
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
ROC-AUC  : 1.0000
FPR      : 0.0000
FNR      : 0.0000

Confusion Matrix [ [TN FP], [FN TP] ]:
[[27  0]
 [ 0 10]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        27
           1     1.0000    1.0

In [43]:
from scipy.stats import chi2_contingency
import pandas as pd

test_results["error_agentic"] = (
    test_results["IS_FRAUD"] != test_results["agentic_pred"]
).astype(int)

cont_table = pd.crosstab(
    test_results["source_template"],
    test_results["error_agentic"]
)

print(cont_table)

chi2, p_value, dof, _ = chi2_contingency(cont_table)

print("Chi2:", chi2)
print("p-value:", p_value)

error_agentic     0  1
source_template       
template1        44  0
template2        37  0
template3        43  0
template4        36  1
template5        39  0
Chi2: 4.427543121010458
p-value: 0.35122565438389747
